# splitting 1000 into validation and test set and remaining into train set 

In [3]:
import random
import os

# ---------- Configuration ----------
INPUT_FILE = 'tokenized_sent_5lakh.txt'
NUM_SENTENCES = 100_000       # number of sentences to read
VALIDATION_SIZE = 1000
TEST_SIZE = 1000
TRAIN_SIZE = NUM_SENTENCES - VALIDATION_SIZE - TEST_SIZE

SAVE_DIR = 'saved_data'
os.makedirs(SAVE_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(SAVE_DIR, 'train_set_1lakh.txt')
VALIDATION_FILE = os.path.join(SAVE_DIR, 'validation_set_1lakh.txt')
TEST_FILE = os.path.join(SAVE_DIR, 'test_set_1lakh.txt')

# ---------- Step 1: Read first 1 lakh sentences ----------
sentences = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= NUM_SENTENCES:
            break
        line = line.strip()
        if line:
            sentences.append(line)

print(f"Total sentences loaded: {len(sentences)}")

# ---------- Step 2: Randomly split indices ----------
all_indices = list(range(NUM_SENTENCES))
test_indices = set(random.sample(all_indices, TEST_SIZE))
remaining_indices = list(set(all_indices) - test_indices)
validation_indices = set(random.sample(remaining_indices, VALIDATION_SIZE))
train_indices = list(set(all_indices) - test_indices - validation_indices)

print(f"Training: {len(train_indices)} | Validation: {len(validation_indices)} | Test: {len(test_indices)}")

# ---------- Step 3: Write sentences to files ----------
with open(TRAIN_FILE, 'w', encoding='utf-8') as train_f, \
     open(VALIDATION_FILE, 'w', encoding='utf-8') as val_f, \
     open(TEST_FILE, 'w', encoding='utf-8') as test_f:

    for i, sentence in enumerate(sentences):
        if i in test_indices:
            test_f.write(sentence + '\n')
        elif i in validation_indices:
            val_f.write(sentence + '\n')
        else:
            train_f.write(sentence + '\n')

print("Data split completed successfully!")


Total sentences loaded: 100000
Training: 98000 | Validation: 1000 | Test: 1000
Data split completed successfully!


In [5]:
import os
import pickle
from collections import Counter

# ---------- Configuration ----------
TRAIN_FILE = 'saved_data/train_set_1lakh.txt'
SAVE_DIR = 'saved_data/ngram_models'
os.makedirs(SAVE_DIR, exist_ok=True)

NGRAMS = [1, 2, 3, 4]  # unigrams, bigrams, trigrams, quadrigrams


# helper function to generate n-grams

In [8]:
def generate_ngrams(sentence, n):
    """
    Generate n-grams from a single sentence with <s> and </s> tokens.
    Returns a generator of tuples.
    """
    tokens = ['<s>'] * (n - 1) + sentence + ['</s>']
    return (tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


# load training set 

In [9]:
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    tokenized_train = [line.strip().split() for line in f if line.strip()]

print(f"Training sentences loaded: {len(tokenized_train)}")


Training sentences loaded: 98000


# n gram model

In [10]:
for n in NGRAMS:
    counts_file = os.path.join(SAVE_DIR, f'{n}gram_counts.pkl')

    # Generate counts
    counts = Counter()
    for sent in tokenized_train:
        counts.update(generate_ngrams(sent, n))

    # Save counts to pickle
    with open(counts_file, 'wb') as f:
        pickle.dump(counts, f)

    print(f"{n}-gram counts generated and saved. Total unique {n}-grams: {len(counts)}")


1-gram counts generated and saved. Total unique 1-grams: 153603
2-gram counts generated and saved. Total unique 2-grams: 785173
3-gram counts generated and saved. Total unique 3-grams: 1121260
4-gram counts generated and saved. Total unique 4-grams: 1231944


# loading n-gram models 

In [11]:
import pickle

SAVE_DIR = 'saved_data/ngram_models'

# ---------- Load unigrams ----------
with open(f'{SAVE_DIR}/1gram_counts.pkl', 'rb') as f:
    unigram_counts = pickle.load(f)

# ---------- Load bigrams ----------
with open(f'{SAVE_DIR}/2gram_counts.pkl', 'rb') as f:
    bigram_counts = pickle.load(f)

# ---------- Load trigrams ----------
with open(f'{SAVE_DIR}/3gram_counts.pkl', 'rb') as f:
    trigram_counts = pickle.load(f)

# ---------- Load quadrigrams ----------
with open(f'{SAVE_DIR}/4gram_counts.pkl', 'rb') as f:
    quadrigram_counts = pickle.load(f)

# ---------- Quick check ----------
print(f"Unigrams loaded: {len(unigram_counts)}")
print(f"Bigrams loaded: {len(bigram_counts)}")
print(f"Trigrams loaded: {len(trigram_counts)}")
print(f"Quadrigrams loaded: {len(quadrigram_counts)}")


Unigrams loaded: 153603
Bigrams loaded: 785173
Trigrams loaded: 1121260
Quadrigrams loaded: 1231944


# import libraries and set paths 

In [12]:
import os
import pickle
from collections import Counter

SAVE_DIR = 'saved_data/ngram_models'
NGRAMS = [1, 2, 3, 4]

# File names for storing probabilities
PROBS_FILES = {
    1: f'{SAVE_DIR}/1gram_probs.pkl',
    2: f'{SAVE_DIR}/2gram_probs.pkl',
    3: f'{SAVE_DIR}/3gram_probs.pkl',
    4: f'{SAVE_DIR}/4gram_probs.pkl',
}


# good turing function

In [13]:
def good_turing_smoothing(counts, vocab_size, n):
    """
    Applies Good-Turing smoothing to counts of n-grams.
    Returns:
        - probs: dict of seen n-grams with GT probability
        - P_unseen: probability for unseen n-grams
    """
    total_seen = sum(counts.values())
    N1 = sum(1 for c in counts.values() if c == 1)  # n-grams seen once

    # Total possible n-grams
    if n == 1:
        total_possible = vocab_size
    else:
        total_possible = vocab_size ** n
    unseen_count = total_possible - len(counts)

    P_unseen = (N1 / total_seen) / unseen_count if unseen_count > 0 else 0

    # Count-of-counts
    count_of_counts = Counter(counts.values())

    probs = {}
    for gram, c in counts.items():
        Nc = count_of_counts[c]
        Ncp1 = count_of_counts.get(c + 1, 0)
        if Nc > 0 and Ncp1 > 0:
            adjusted_count = (c + 1) * (Ncp1 / Nc)
        else:
            adjusted_count = c
        probs[gram] = adjusted_count / total_seen

    return probs, P_unseen


In [14]:
# Example if counts are already loaded:
# unigram_counts, bigram_counts, trigram_counts, quadrigram_counts
# (assuming you have loaded them as in previous step)
counts_dict = {
    1: unigram_counts,
    2: bigram_counts,
    3: trigram_counts,
    4: quadrigram_counts
}

# Vocabulary size
vocab = set(word for sent in tokenized_train for word in sent)
V = len(vocab)
print(f"Vocabulary size: {V}")


Vocabulary size: 153602


# compute good turing probabilities for all n-grams 

In [15]:
probs_dict = {}  # store in memory for immediate use

for n in NGRAMS:
    probs_file = PROBS_FILES[n]

    # Load if already exists
    if os.path.exists(probs_file):
        print(f"Loading existing {n}-gram Good-Turing probabilities...")
        with open(probs_file, 'rb') as f:
            data = pickle.load(f)
        probs_dict[n] = data
    else:
        print(f"Computing Good-Turing for {n}-grams...")
        counts = counts_dict[n]
        probs, P_unseen = good_turing_smoothing(counts, V, n)
        probs_dict[n] = {'probs': probs, 'P_unseen': P_unseen}

        # Save to pickle
        with open(probs_file, 'wb') as f:
            pickle.dump(probs_dict[n], f)
        print(f"{n}-gram Good-Turing probabilities saved.")

    print(f"{n}-gram example probs: {list(probs_dict[n]['probs'].items())[:3]}")
    print(f"{n}-gram unseen probability: {probs_dict[n]['P_unseen']}\n")


Computing Good-Turing for 1-grams...
1-gram Good-Turing probabilities saved.
1-gram example probs: [(('આ',), 0.011392495296329444), (('વીડિયો',), 0.0009050920608953427), (('જુઓ:',), 1.0010051860187554e-05)]
1-gram unseen probability: 0

Computing Good-Turing for 2-grams...
2-gram Good-Turing probabilities saved.
2-gram example probs: [(('<s>', 'આ'), 0.004881160764367167), (('આ', 'વીડિયો'), 3.476199951021183e-05), (('વીડિયો', 'જુઓ:'), 2.138369400967727e-06)]
2-gram unseen probability: 2.043827421958003e-11

Computing Good-Turing for 3-grams...
3-gram Good-Turing probabilities saved.
3-gram example probs: [(('<s>', '<s>', 'આ'), 0.004881160764367167), (('<s>', 'આ', 'વીડિયો'), 2.2810912083663052e-05), (('આ', 'વીડિયો', 'જુઓ:'), 2.011258233819529e-06)]
3-gram unseen probability: 2.1002640595871998e-16

Computing Good-Turing for 4-grams...
4-gram Good-Turing probabilities saved.
4-gram example probs: [(('<s>', '<s>', '<s>', 'આ'), 0.004881160764367167), (('<s>', '<s>', 'આ', 'વીડિયો'), 2.680769

# what this will do 

probs_dict[1]  # unigrams

probs_dict[2]  # bigrams

probs_dict[3]  # trigrams

probs_dict[4]  # quadrigrams

each dictionary has
{
    'probs': {('I',): 0.00045, ('love',): 0.00012, ...},
    'P_unseen': 1.2e-06
}


# compute probabilities in validation and test sentences 

In [18]:
import os
import pickle
import math

SAVE_DIR = 'saved_data/ngram_models'
VALIDATION_FILE = 'saved_data/validation_set_1lakh.txt'
TEST_FILE = 'saved_data/test_set_1lakh.txt'

PROB_SAVE_DIR = 'saved_data/sentence_probs'
os.makedirs(PROB_SAVE_DIR, exist_ok=True)


# load sentences 

In [19]:
def load_sentences(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip().split() for line in f if line.strip()]

validation_sentences = load_sentences(VALIDATION_FILE)
test_sentences = load_sentences(TEST_FILE)

print(f"Validation sentences: {len(validation_sentences)}")
print(f"Test sentences: {len(test_sentences)}")


Validation sentences: 1000
Test sentences: 1000


# function to compute log probabilites 

In [21]:
def sentence_log_probability(sentence, ngram_probs, n, vocab_size):
    """
    Compute log-probability of a sentence using n-gram Good-Turing model.
    """
    probs = ngram_probs['probs']
    P_unseen = ngram_probs['P_unseen']
    
    tokens = ['<s>'] * (n-1) + sentence + ['</s>']
    log_prob = 0.0
    
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        p = probs.get(ngram, P_unseen)
        # Avoid log(0)
        if p > 0:
            log_prob += math.log(p)
        else:
            log_prob += math.log(1e-12)  # very small value if somehow 0
    
    return log_prob


# compute log probabilites 

In [22]:
# Choose which n-gram model to use (1,2,3,4)
n = 2  # example: bigram
vocab_size = len(vocab)
ngram_model = probs_dict[n]

# Compute validation sentence probabilities
validation_probs = [(sentence, sentence_log_probability(sentence, ngram_model, n, vocab_size))
                    for sentence in validation_sentences]

# Compute test sentence probabilities
test_probs = [(sentence, sentence_log_probability(sentence, ngram_model, n, vocab_size))
              for sentence in test_sentences]

print("First 3 validation sentences with log-probabilities:")
for sent, prob in validation_probs[:3]:
    print(sent, prob)

print("First 3 test sentences with log-probabilities:")
for sent, prob in test_probs[:3]:
    print(sent, prob)


First 3 validation sentences with log-probabilities:
['૭પ)', '(નિવૃત', 'જીઇબી', 'એન્જીનીયર)', 'તે', 'હીરેશનભાઇ,', 'ચેતનભાઇ', 'ના', 'પિતાશ્રીનું', 'તા.'] -242.4451878541599
['ભાવનગરનાં', 'નિલમબાગ', 'પોલીસ', 'મથકમાં', 'અધેવાડામાં', 'રહેતા', 'ઇન્દ્વજીતસિંહ', 'ઉર્ફે', 'ઇનો', 'વિક્રમસિંહ', 'ગોહિલ', 'કાચા', 'કામના', 'કેદી', 'તરીકે', 'ભાવનગર', 'જેલમાં', 'સજા', 'ભોગી', 'રહ્યો', 'હતો.'] -456.88554330734075
['ઓટોરિક્ષા', 'ચાલકોને', 'નવી', 'ઓળખ', 'મળી', 'ગઈ', 'છે.'] -127.39075517153528
First 3 test sentences with log-probabilities:
['પ્રણવ', 'મુખરજીએ', 'હોસ્પિટલની', 'તકતીનું', 'અનાવરણ', 'કર્યું', 'તહું.'] -157.78652848500707
['9', 'ટકા', 'સુધીનો', 'વધારો', 'નોંધાયો', 'હતો.'] -74.18052238734397
['.?'] -23.16572263027285


Closer to 0 → the sentence is more probable.

More negative → the sentence is less probable.

# save the sentence probabilites with pickle 

In [23]:
with open(f'{PROB_SAVE_DIR}/validation_sentence_probs_{n}gram.pkl', 'wb') as f:
    pickle.dump(validation_probs, f)

with open(f'{PROB_SAVE_DIR}/test_sentence_probs_{n}gram.pkl', 'wb') as f:
    pickle.dump(test_probs, f)

print(f"Validation and test sentence probabilities for {n}-gram model saved successfully!")


Validation and test sentence probabilities for 2-gram model saved successfully!


# what will it store 
(['I', 'love', 'python'], -23.45)


# 3 : show a table with top 100 frequencies for bigram model 

# cmopute Nc for counts 

In [24]:
from collections import Counter
import pandas as pd

# Example: use bigram counts
counts = bigram_counts  # replace with unigram_counts, trigram_counts, etc.

# Count-of-counts: Nc = number of n-grams that appear exactly c times
count_of_counts = Counter(counts.values())

# Sort by c
sorted_counts = sorted(count_of_counts.items())  # list of tuples (c, Nc)


# Compute Good-Turing adjusted counts (C)*

In [25]:
C_star = []

for c, Nc in sorted_counts[:100]:  # top 100 frequencies
    Ncp1 = count_of_counts.get(c+1, 0)
    if Nc > 0 and Ncp1 > 0:
        c_star = (c+1) * (Ncp1 / Nc)
    else:
        c_star = c
    C_star.append((c, Nc, c_star))


In [26]:
df_gt = pd.DataFrame(C_star, columns=['c', 'Nc', 'C*'])
pd.set_option('display.float_format', '{:.4f}'.format)
df_gt


,c,Nc,C*
0,1,669678,0.1809
1,2,60586,0.9947
2,3,20089,1.9507
3,4,9797,2.9698
4,5,5819,3.8739
...,...,...,...
95,96,5,97.0000
96,97,5,98.0000
97,98,5,79.2000
98,99,4,225.0000


# 4 : Implement deleted interpolated smoothing technique for the quadrigram model and find the best parameters.

In [ ]:
import pickle
import collections
import math
import csv
import json
from pathlib import Path
from tqdm import tqdm 

def load_ngram_counts(n, folder="."):
    file_name = Path(folder) / f"ngram_counts_{n}gram.pkl"
    with open(file_name, "rb") as f:
        counts = pickle.load(f)
    return counts

def get_total_unigram_tokens(unigrams):
    return sum(unigrams.values())

def safe_div(num, den):
    if den <= 0:
        return 0.0
    return num / den

def compute_deleted_interpolation_lambdas(unigrams, bigrams, trigrams, quadrigrams):
    
    acc = [0.0, 0.0, 0.0, 0.0]  
    # Unigrams keys: (w,), bigrams: (w1,w2), trigrams: (w1,w2,w3), quadrigrams: (w1,w2,w3,w4)

    N_unigram_tokens = get_total_unigram_tokens(unigrams)

    for quad, c_quad in tqdm(quadrigrams.items(), desc="Deleted-Interpolation", unit="quad"):
        c = c_quad
        if c <= 0:
            continue

        w1, w2, w3, w4 = quad

        # 4-gram 
        context4 = (w1, w2, w3)
        c_context4 = trigrams.get(context4, 0)
        p4 = 0.0
        if c_context4 - 1 > 0:
            p4 = safe_div(c_quad - 1, c_context4 - 1)

        # 3-gram:
        tri = (w2, w3, w4)
        context3 = (w2, w3)
        c_tri = trigrams.get(tri, 0)
        c_context3 = bigrams.get(context3, 0)
        p3 = 0.0
        if c_context3 - 1 > 0:
            p3 = safe_div(c_tri - 1, c_context3 - 1)

        # 2-gram:
        bi = (w3, w4)
        context2 = (w3,)
        c_bi = bigrams.get(bi, 0)
        c_context2 = unigrams.get(context2, 0)
        p2 = 0.0
        if c_context2 - 1 > 0:
            p2 = safe_div(c_bi - 1, c_context2 - 1)

        # 1-gram:
        unigram = (w4,)
        c_uni = unigrams.get(unigram, 0)
        p1 = 0.0
        if N_unigram_tokens - 1 > 0:
            p1 = safe_div(c_uni - 1, N_unigram_tokens - 1)

        ps = [p1, p2, p3, p4]
        best_k = max(range(4), key=lambda k: ps[k])  
        acc[best_k] += c  # weight by count (or you can add 1 if preferred)

    total = sum(acc)
    if total == 0:
        # fallback to uniform
        lambdas = [0.25, 0.25, 0.25, 0.25]
    else:
        lambdas = [a / total for a in acc]

    return lambdas

# ---------------- Interpolated probability ----------------
def interpolated_prob(w_prev3, w_prev2, w_prev1, w, unigrams, bigrams, trigrams, quadrigrams, lambdas):
    # compute MLE conditional probabilities (non-deleted) at each order:
    # P4 = c(w_prev3,w_prev2,w_prev1,w)/c(w_prev3,w_prev2,w_prev1)
    c4 = quadrigrams.get((w_prev3, w_prev2, w_prev1, w), 0)
    c_context4 = trigrams.get((w_prev3, w_prev2, w_prev1), 0)
    p4 = safe_div(c4, c_context4)

    c3 = trigrams.get((w_prev2, w_prev1, w), 0)
    c_context3 = bigrams.get((w_prev2, w_prev1), 0)
    p3 = safe_div(c3, c_context3)

    c2 = bigrams.get((w_prev1, w), 0)
    c_context2 = unigrams.get((w_prev1,), 0)
    p2 = safe_div(c2, c_context2)

    c1 = unigrams.get((w,), 0)
    N = get_total_unigram_tokens(unigrams)
    p1 = safe_div(c1, N)

    lamb1, lamb2, lamb3, lamb4 = lambdas
    return lamb1*p1 + lamb2*p2 + lamb3*p3 + lamb4*p4

def sentence_logprob_interpolated(sentence, unigrams, bigrams, trigrams, quadrigrams, lambdas, n=4):
    tokens = sentence.strip().split()
    # pad with start tokens
    tokens = ["<s>"]*(n-1) + tokens + ["</s>"]
    logp = 0.0
    word_count = 0
    for i in range(n-1, len(tokens)):
        w = tokens[i]
        w_prev1 = tokens[i-1] if i-1 >= 0 else "<s>"
        w_prev2 = tokens[i-2] if i-2 >= 0 else "<s>"
        w_prev3 = tokens[i-3] if i-3 >= 0 else "<s>"
        prob = interpolated_prob(w_prev3, w_prev2, w_prev1, w,
                                 unigrams, bigrams, trigrams, quadrigrams, lambdas)
        if prob <= 0:
            prob = 1e-12
        logp += math.log(prob)
        word_count += 1
    return logp, word_count

def evaluate_perplexity(file_path, unigrams, bigrams, trigrams, quadrigrams, lambdas):
    total_logprob = 0.0
    total_words = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            lp, wc = sentence_logprob_interpolated(s, unigrams, bigrams, trigrams, quadrigrams, lambdas)
            total_logprob += lp
            total_words += wc

    # perplexity = exp(- total_logprob / total_words)
    avg_neg_log_likelihood = - total_logprob / total_words
    perp = math.exp(avg_neg_log_likelihood)
    return perp

# ---------------- Main ----------------
if _name_ == "_main_":
    # Load counts
    unigrams = load_ngram_counts(1)
    bigrams = load_ngram_counts(2)
    trigrams = load_ngram_counts(3)
    quadrigrams = load_ngram_counts(4)

    # compute lambdas
    lambdas = compute_deleted_interpolation_lambdas(unigrams, bigrams, trigrams, quadrigrams)
    print("Deleted-interpolation lambdas (lambda1..lambda4):", lambdas)

    # Evaluate on validation set and optionally on test
    val_file = "validation.txt"
    test_file = "test.txt"

    print("Evaluating on validation set...")
    val_ppl = evaluate_perplexity(val_file, unigrams, bigrams, trigrams, quadrigrams, lambdas)
    print(f"Validation Perplexity (interpolated quadrigram): {val_ppl:.4f}")

    print("Evaluating on test set...")
    test_ppl = evaluate_perplexity(test_file, unigrams, bigrams, trigrams, quadrigrams, lambdas)
    print(f"Test Perplexity (interpolated quadrigram): {test_ppl:.4f}")

    # Save lambdas
    with open("quad_interpolation_lambdas.json", "w", encoding="utf-8") as out:
        json.dump({"lambdas": lambdas}, out, indent=2)
    print("Lambdas saved to quad_interpolation_lambdas.json")

Deleted-Interpolation: 100%|██████████| 11619213/11619213 [00:48<00:00, 241725.84quad/s]<br>
Deleted-interpolation lambdas (lambda1..lambda4): [0.40355213390901956, 0.38137513613787494, 0.15748241207594238, 0.0575903178771631]<br>
Evaluating on validation set<br>
Validation Perplexity (interpolated quadrigram): 1921.<br>
Evaluating on test set...<br>
Test Perplexity (interpolated quadrigram): 2188.2764<br>
Lambdas saved to quad_interpolation_lambdas.json<br>